# Praca z tekstem w pandas — `.str`, regex i triki z `lambda`

**Problem:** pandas ma bogaty accessor `.str` pokrywający większość zadań tekstowych bez `lambda` — ale różnica między pokrewnymi metodami (`split` vs `partition`, `find` vs `index`, `replace` z regex vs bez) bywa nieoczywista, a sięgnięcie po `lambda`/`apply()` tam, gdzie `.str` by wystarczyło, kosztuje realną wydajność.

**Porównanie:**
- `.str.*` — wektoryzowane, bezpiecznie obsługują `NaN` (zwracają `NaN`, nie rzucają błędu), zwykle szybsze.
- `.apply(lambda x: ...)` — elastyczne, ale wolniejsze i **nie** obsługują `NaN` automatycznie — trzeba to ogarnąć samemu.

**Kiedy stosować `lambda`:** gdy logika łączy kilka kolumn naraz (`axis=1`), zależy od warunku, którego nie da się wyrazić jedną metodą `.str`, albo gdy potrzebna mapa/słownik zależny od wartości w innej kolumnie. W każdym innym przypadku najpierw szukaj gotowej metody `.str`.

## Setup

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "product_code": ["PRD-2026-00123", "PRD-2026-00456", "SVC-2025-00789", "PRD-2026-00234"],
    "raw_name": ["  Router TP-Link AC1200 ", "SWITCH netgear 8-port", "cable HDMI 2m", "  Access Point UBIQUITI  "],
    "email": ["jan.kowalski@firma.pl", "anna_nowak@firma.pl", "piotr.wisniewski@firma.pl", "ewa-kowal@firma.pl"],
    "tags": ["electronics,networking,sale", "electronics,networking", "cables,accessories", "electronics,wifi,new"],
    "file_path": ["data/region_north/report_2026.csv", "data/region_south/report_2026.csv",
                  "data/region_east/summary.csv", "data/region_west/report_2026_v2.csv"],
})
df

## Sekcja 1 — Czyszczenie: `strip`, wielkość liter, długość

**Uwaga na `.str.title()`:** kapitalizuje literę po KAŻDYM znaku niebędącym literą, nie tylko po spacji — `"8-port"` staje się `"8-Port"`, a `"2m"` → `"2M"`. Wygląda niewinnie, ale przy nazwach z myślnikami/cyframi potrafi zaskoczyć.

In [ ]:
df["raw_name"].str.strip()

In [ ]:
clean = df["raw_name"].str.strip()
print(clean.str.lower())
print(clean.str.upper())
print(clean.str.title())      # uwaga: "8-port" -> "8-Port", "2m" -> "2M"
print(clean.str.capitalize()) # tylko pierwsza litera calego stringa, reszta lower
print(clean.str.len())

## Sekcja 2 — Dzielenie tekstu: `split`

- bez argumentów `n=` — dzieli na wszystkie wystąpienia separatora, zwraca listy o różnej długości.
- `n=` — ogranicza liczbę podziałów (przydatne np. przy adresie e-mail, żeby rozdzielić tylko na pierwszej kropce).
- `rsplit` — dzieli od końca; z `n=1` daje "wszystko oprócz ostatniego fragmentu" + "ostatni fragment" — idealne do rozdzielenia ścieżki na katalog i nazwę pliku.
- `expand=True` — zamiast list w jednej kolumnie, od razu rozbija wynik na osobne kolumny `DataFrame`.

In [ ]:
df["product_code"].str.split("-")

In [ ]:
# n= ogranicza liczbe podzialow - tu tylko na pierwszej kropce w mailu
df["email"].str.split(".", n=1)

In [ ]:
# rsplit z n=1 - katalog + nazwa pliku, licząc od KOŃCA ścieżki
df["file_path"].str.rsplit("/", n=1)

In [ ]:
# expand=True - od razu jako osobne kolumny DataFrame, nie listy
df["product_code"].str.split("-", expand=True)

## Sekcja 3 — Tekst przed/po separatorze: `partition` / `rpartition`

`partition()` dzieli **tylko na pierwszym** wystąpieniu separatora i zawsze zwraca dokładnie 3 kolumny: `(przed, separator, po)`. `rpartition()` robi to samo, ale od **ostatniego** wystąpienia. W przeciwieństwie do `split`, liczba kolumn wyniku jest zawsze stała — nie trzeba martwić się o `expand=True` ani o różną liczbę fragmentów (patrz też Pułapka 2).

In [ ]:
df["email"].str.partition("@")

In [ ]:
# rpartition na ścieżce pliku - dokładnie ten sam efekt co rsplit(n=1), inny zapis
df["file_path"].str.rpartition("/")

## Sekcja 4 — Wyciąganie fragmentów: `extract` / `extractall`

`extract()` z grupami regex (`(...)`)  zwraca jedno dopasowanie na wiersz — po jednej kolumnie na grupę. **Nazwane grupy** (`(?P<nazwa>...)`) od razu nadają czytelne nazwy kolumn zamiast `0, 1, 2`. `extractall()` zwraca **wszystkie** dopasowania w tekście (nie tylko pierwsze), z dodatkowym poziomem indeksu numerującym kolejne trafienia.

In [ ]:
df["product_code"].str.extract(r"(?P<prefix>[A-Z]+)-(?P<year>\d+)-(?P<number>\d+)")

In [ ]:
codes_in_text = pd.Series(["kod: A1, kod: B2, kod: C3"])
codes_in_text.str.extractall(r"kod: (?P<kod>\w\d)")

## Sekcja 5 — Tylko cyfry / tylko litery

Najprostsze podejście: regex `replace()` usuwający wszystko, co NIE pasuje do wzorca (`[^0-9]` = "nie-cyfra", `[0-9]` = "cyfra" do usunięcia).

In [ ]:
print("Tylko cyfry:")
print(df["product_code"].str.replace(r"[^0-9]", "", regex=True))

print("\nTylko litery (usuwamy cyfry i myślniki):")
print(df["product_code"].str.replace(r"[^A-Za-z]", "", regex=True))

## Sekcja 6 — Podmiana tekstu: `replace`

`Series.str.replace()` (podmiana fragmentu tekstu, z regexem lub bez) to inna metoda niż `Series.replace()` (podmiana **całych wartości**, jak w Sekcji o duplikatach z innych notatek) — łatwo je pomylić po samej nazwie.

In [ ]:
clean = df["raw_name"].str.strip()

# str.replace - podmiana FRAGMENTU tekstu
print(clean.str.replace("TP-Link", "TPLink"))

# case=False - podmiana niezależna od wielkości liter (wymaga regex=True)
print(clean.str.replace("router", "ROUTER", case=False, regex=True))

# Series.replace (BEZ .str) - podmiana CAŁEJ wartości komórki, nie fragmentu
print(clean.replace({"cable HDMI 2m": "HDMI Cable 2m"}))

## Sekcja 7 — Pozycja wystąpienia: `find`, `rfind`, `count`

`.str.find()` to odpowiednik Pythonowego `str.find()`, nie `str.index()` — przy braku dopasowania zwraca `-1` zamiast rzucać wyjątek. Bezpieczniejsze w kontekście wektoryzowanym, gdzie różne wiersze mogą, ale nie muszą zawierać szukanego fragmentu.

In [ ]:
print(df["email"].str.find("@"))       # pozycja pierwszego wystąpienia
print(df["email"].str.count(r"\."))    # liczba wystąpień

print("\nRóżnica find vs Python index() przy braku dopasowania:")
print("pd.Series(['brak_znaku']).str.find('@') ->", pd.Series(["brak_znaku"]).str.find("@").tolist(), "(bez błędu)")
try:
    "brak_znaku".index("@")
except ValueError as e:
    print(f"czysty Python str.index('@') -> błąd: {e}")

## Sekcja 8 — Sprawdzanie zawartości: `isalpha`, `isdigit`, `contains`, `startswith`

Cała rodzina `is*` sprawdza CAŁY string (nie pojedynczy znak) — `"abc123"` to ani `isalpha`, ani `isdigit`, bo miesza litery i cyfry.

In [ ]:
words = pd.Series(["Warszawa", "12345", "abc123", "  ", "ABC"])
print("isalpha:", words.str.isalpha().tolist())
print("isdigit:", words.str.isdigit().tolist())
print("isalnum:", words.str.isalnum().tolist())  # litery LUB cyfry, mieszanka OK
print("isupper:", words.str.isupper().tolist())

In [ ]:
print(df["email"].str.contains("firma.pl"))       # uwaga: '.' w contains to też regex, patrz Pułapka 4
print(df["product_code"].str.startswith("PRD"))
print(df["file_path"].str.endswith(".csv"))

## Sekcja 9 — `removeprefix` / `removesuffix`

Bezpieczny odpowiednik ręcznego `str[len(prefix):]` — jeśli string NIE zaczyna się od podanego prefiksu, zwraca go bez zmian zamiast obcinać coś przypadkiem.

In [ ]:
# 'SVC-2025-00789' nie zaczyna się od 'PRD-' -> zostaje bez zmian, bez błędu
df["product_code"].str.removeprefix("PRD-")

In [ ]:
pd.Series(["report_2026.csv", "summary.csv"]).str.removesuffix(".csv")

## Sekcja 10 — Łączenie tekstu: `str.cat`, `join`, `agg` + `", ".join`

In [ ]:
# str.cat - sklejenie DWÓCH kolumn wiersz-po-wierszu
df["product_code"].str.cat(df["raw_name"].str.strip(), sep=" | ")

In [ ]:
# .str.get(i) - dostęp do elementu listy po split, BEZ lambda (i=-1 to ostatni element)
split_tags = df["tags"].str.split(",")
print("Pierwszy tag:", split_tags.str.get(0).tolist())
print("Ostatni tag:", split_tags.str.get(-1).tolist())

# .str.join - sklejenie LISTY z powrotem w jeden string z innym separatorem
split_tags.str.join(" / ")

In [ ]:
# agg + ", ".join po groupby - scalenie tekstów W OBRĘBIE grupy
sample = pd.DataFrame({"category": ["A", "A", "B", "B"], "tag": ["x", "y", "z", "w"]})
sample.groupby("category")["tag"].agg(", ".join)

## Sekcja 11 — `lambda`: kiedy warto, a kiedy szkodzi

### Benchmark: `.str` vs `.apply(lambda)` dla tej samej operacji

In [ ]:
import time

rng = np.random.default_rng(5)
n = 200_000
words_big = pd.Series(rng.choice(["Warszawa", "Krakow", "Gdansk", "Wroclaw", "Poznan"], n))

start = time.perf_counter()
words_big.str.upper()
t_str = time.perf_counter() - start

start = time.perf_counter()
words_big.apply(lambda x: x.upper())
t_lambda = time.perf_counter() - start

print(f".str.upper():                {t_str:.4f}s")
print(f".apply(lambda x: x.upper()): {t_lambda:.4f}s")
print(f"Różnica: {t_lambda / t_str:.1f}x wolniej")

### Kiedy `lambda` jest właściwym wyborem: logika łącząca kilka kolumn naraz

Tego typu warunkowego budowania tekstu z kilku źródeł (słownik + kilka kolumn + formatowanie) nie da się łatwo wyrazić jedną metodą `.str` — tu `lambda` z `axis=1` jest uzasadniona, nie antywzorcem.

In [ ]:
orders = pd.DataFrame({
    "prefix": ["PRD", "SVC"],
    "region": ["North", "South"],
    "sales": [1200, 800],
})
label_map = {"PRD": "Produkt", "SVC": "Usluga"}

orders["label"] = orders.apply(
    lambda row: f"{label_map[row['prefix']]} ({row['region']}): {row['sales']} PLN", axis=1
)
orders

## Sekcja 12 — Dopełnianie: `pad`, `ljust`, `rjust`, `center`, `zfill`

Przydatne przy wyrównywaniu tekstu do stałej szerokości (np. eksport do formatu o stałej długości pola) albo przy numerach/kodach wymagających zer wiodących.

In [ ]:
cities = pd.Series(["Warszawa", "Lodz", "Gdansk"])

print(cities.str.ljust(12, fillchar="."))   # wyrównanie do LEWEJ, dopełnienie z prawej
print(cities.str.rjust(12, fillchar="."))   # wyrównanie do PRAWEJ, dopełnienie z lewej
print(cities.str.center(12, fillchar="."))  # wyśrodkowanie

In [ ]:
# zfill - dopełnienie zerami wiodącymi, typowe dla ID/kodów o stałej długości
order_numbers = pd.Series(["7", "42", "123"])
order_numbers.str.zfill(5)

## Sekcja 13 — Wycinanie po pozycji: `slice`, `slice_replace`, `repeat`

`.str.slice()` to wektoryzowany odpowiednik zwykłego `string[a:b]` — przydatny, gdy struktura tekstu jest stała pozycyjnie (np. pierwsze 3 znaki to zawsze prefiks kategorii), a nie oddzielona separatorem (wtedy lepszy `split`/`partition`, Sekcje 2–3).

In [ ]:
codes_fixed_width = pd.Series(["PRD20260123", "SVC20250789"])

print("Znaki 0-3 (prefiks):", codes_fixed_width.str.slice(0, 3).tolist())
print("Znaki 3-7 (rok):    ", codes_fixed_width.str.slice(3, 7).tolist())
print()

# slice_replace - podmiana fragmentu WSKAZANEGO POZYCYJNIE, nie przez dopasowanie wzorca
print(codes_fixed_width.str.slice_replace(0, 3, "XXX"))

In [ ]:
# repeat - powtórzenie CAŁEGO stringa n razy (rzadkie, ale przydatne np. do separatorów wizualnych)
pd.Series(["ab", "cd"]).str.repeat(3)

## Sekcja 14 — `match` vs `fullmatch` vs `contains`: trzy różne poziomy dopasowania

Wszystkie trzy przyjmują ten sam wzorzec regex, ale różnią się tym, JAK DUŻO tekstu musi pasować:
- `contains` — wzorzec może wystąpić GDZIEKOLWIEK w tekście.
- `match` — dopasowanie musi zaczynać się od POCZĄTKU tekstu (ale nie musi go w całości "skonsumować").
- `fullmatch` — CAŁY tekst musi dokładnie pasować do wzorca, od początku do końca.

In [ ]:
codes_to_validate = pd.Series(["PRD-2026-00123", "invalid-code", "SVC-2025-00789"])
pattern = r"[A-Z]{3}-\d{4}-\d{5}"

print("contains:  ", codes_to_validate.str.contains(pattern, regex=True).tolist())
print("match:     ", codes_to_validate.str.match(pattern).tolist())
print("fullmatch: ", codes_to_validate.str.fullmatch(pattern).tolist())

In [ ]:
# Różnica match vs fullmatch ujawnia się dopiero przy DODATKOWYM tekście na końcu
with_extra_suffix = pd.Series(["PRD-2026-00123-extra_dopisek"])

print(f"match:     {with_extra_suffix.str.match(pattern).tolist()}  (pasuje - dopasowanie zaczyna się poprawnie)")
print(f"fullmatch: {with_extra_suffix.str.fullmatch(pattern).tolist()}  (NIE pasuje - jest 'ogon' po dopasowaniu)")

## Sekcja 15 — `replace()` z funkcją zamiast tekstu: dynamiczna podmiana

Gdy podmiana zależy od TEGO, co zostało dopasowane (nie jest stałym tekstem), `str.replace()` przyjmuje funkcję zamiast stringa. Funkcja dostaje obiekt `re.Match` i musi zwrócić string — pozwala np. przeliczyć dopasowaną liczbę w locie.

In [ ]:
prices_text = pd.Series(["cena: 100zl", "cena: 250zl"])

# Doliczenie 23% VAT do dopasowanej kwoty - wartość podmiany zależy od dopasowania, nie jest stała
with_vat = prices_text.str.replace(
    r"(\d+)zl",
    lambda m: f"{int(m.group(1)) * 1.23:.2f}zl (z VAT)",
    regex=True,
)
with_vat

## Sekcja 16 — `split()` z regexem: kilka możliwych separatorów naraz

Gdy dane mają niespójne separatory (typowe przy ręcznie wprowadzanych danych — raz przecinek, raz średnik, raz spacja), wzorzec regex w `split()` obsługuje wszystkie naraz w jednym wywołaniu.

In [ ]:
messy_separators = pd.Series(["a,b;c d", "x; y,z"])
messy_separators.str.split(r"[,; ]+")  # przecinek, średnik LUB spacja, jedno lub więcej pod rząd

## Sekcja 17 — `extract()` z opcjonalną grupą

`(?:...)` tworzy grupę NIE-przechwytującą (do grupowania samej logiki regex, bez tworzenia kolumny wyniku), a `?` po niej czyni ją opcjonalną. Przydatne, gdy część tekstu bywa, ale nie musi występować — np. opcjonalny prefiks numeru telefonu.

In [ ]:
phones = pd.Series(["+48 123456789", "123456789"])

# (?:...)? - opcjonalny, nie-przechwytujący blok z prefiksem kraju
phones.str.extract(r"(?:\+(?P<prefix>\d{2}) )?(?P<number>\d{9})")

## Sekcja 18 — `translate()`: mapowanie znak-na-znak

Szybsza alternatywa dla łańcucha wielu `str.replace()`, gdy zamieniasz POJEDYNCZE ZNAKI na inne pojedyncze znaki — klasyczny przykład: usuwanie polskich znaków diakrytycznych (np. pod nazwy plików/identyfikatory bez ogonków).

In [ ]:
polish_chars = "ąćęłńóśźżĄĆĘŁŃÓŚŹŻ"
ascii_chars = "acelnoszzACELNOSZZ"
translation_map = str.maketrans(polish_chars, ascii_chars)

cities_pl = pd.Series(["Łódź", "Gdańsk", "Wrocław"])
cities_pl.str.translate(translation_map)

## Sekcja 19 — Łańcuchowanie metod `.str` (method chaining)

Każda metoda `.str.*` zwraca `Series`, więc kolejne wywołania można doklejać jedno po drugim — zamiast zapisywać wynik pośredni do zmiennej po każdym kroku. Czytelniejsze przy kilku prostych, następujących po sobie przekształceniach.

In [ ]:
raw_names = pd.Series(["  Router TP-Link  ", "SWITCH netgear"])

slug = (
    raw_names
    .str.strip()
    .str.lower()
    .str.replace("-", "_", regex=False)
    .str.replace(" ", "_", regex=False)
)
slug

## Sekcja 12 — Pułapki

### Pułapka 1 — `split(expand=True)` z różną liczbą fragmentów tworzy `NaN`

Jeśli liczba wystąpień separatora różni się między wierszami, brakujące kolumny wypełniają się `NaN` — nie ma żadnego ostrzeżenia, że któryś wiersz miał inny kształt niż pozostałe.

In [ ]:
paths = pd.Series(["a/b/c", "a/b", "a/b/c/d"])
paths.str.split("/", expand=True)

### Pułapka 2 — `partition` zawsze zwraca 3 elementy, nawet gdy separatora brak

Przy braku separatora `partition` NIE rzuca błędu ani nie zwraca `NaN` — środkowa i ostatnia kolumna to po prostu puste stringi, a cały oryginalny tekst ląduje w pierwszej kolumnie. Trzeba to świadomie sprawdzić, jeśli "brak separatora" ma dla Ciebie inne znaczenie niż "pusty wynik".

In [ ]:
s = pd.Series(["jan@firma.pl", "brak_separatora"])
s.str.partition("@")

### Pułapka 3 — `NaN` w kolumnie: `.str` radzi sobie, `.apply(lambda)` rzuca błąd

To kolejny, obok wydajności, powód, żeby domyślnie sięgać po `.str` zamiast `lambda` — `.str` metody automatycznie zwracają `NaN` dla brakujących wartości, `lambda` wymaga ręcznej obsługi (`x.upper() if pd.notna(x) else x`).

In [ ]:
s_nan = pd.Series(["Warszawa", np.nan, "Krakow"])

print("str.upper() - działa mimo NaN:")
print(s_nan.str.upper())

print("\napply(lambda x: x.upper()) - wybucha na NaN:")
try:
    s_nan.apply(lambda x: x.upper())
except AttributeError as e:
    print(f"Błąd: {e}")

### Pułapka 4 — `replace()` domyślnie traktuje wzorzec jako regex — `"."` to "dowolny znak"

`str.replace(".", "_")` **nie** zamienia tylko kropek — `"."` w regexie oznacza "dowolny znak", więc zamienia KAŻDY znak w stringu. Ten sam problem dotyczy `.str.contains()` z kropką w szukanym fragmencie. Zabezpieczenie: `regex=False` dla dosłownego dopasowania, albo escape (`r"\."`).

In [ ]:
files = pd.Series(["plik.csv", "raport.csv"])

print("Domyślny regex=True - '.' oznacza 'dowolny znak', zamienia WSZYSTKO:")
print(files.str.replace(".", "_", regex=True))

print("\nPoprawnie: regex=False - dosłowna kropka:")
print(files.str.replace(".", "_", regex=False))

## Podsumowanie

| Zadanie | Rozwiązanie |
|---|---|
| Usunięcie białych znaków z brzegów | `.str.strip()` / `lstrip()` / `rstrip()` |
| Wielkość liter | `.str.lower()` / `upper()` / `title()` (uwaga: kapitalizuje po `-`/cyfrach) / `capitalize()` |
| Podział na wszystkie fragmenty | `.str.split(sep)` |
| Podział z limitem / od końca | `.str.split(sep, n=)` / `.str.rsplit(sep, n=)` |
| Podział na osobne kolumny | `.str.split(sep, expand=True)` |
| Tekst przed/po PIERWSZYM separatorze | `.str.partition(sep)` |
| Tekst przed/po OSTATNIM separatorze | `.str.rpartition(sep)` |
| Wyciągnięcie fragmentu wg wzorca | `.str.extract(r"(?P<nazwa>...)")` |
| Wszystkie dopasowania wzorca | `.str.extractall(r"...")` |
| Tylko cyfry / tylko litery | `.str.replace(r"[^0-9]", "", regex=True)` / `r"[^A-Za-z]"` |
| Podmiana fragmentu tekstu | `.str.replace(stary, nowy, regex=..., case=...)` |
| Podmiana całej wartości komórki | `.replace({stara_wartość: nowa_wartość})` (bez `.str`) |
| Pozycja pierwszego wystąpienia (bezpieczna) | `.str.find(fragment)` (zwraca `-1`, nie rzuca błędu) |
| Liczba wystąpień | `.str.count(wzorzec)` |
| Czy CAŁY string to litery/cyfry | `.str.isalpha()` / `isdigit()` / `isalnum()` |
| Czy zawiera / zaczyna / kończy się na | `.str.contains()` / `startswith()` / `endswith()` |
| Bezpieczne usunięcie prefiksu/sufiksu | `.str.removeprefix()` / `.str.removesuffix()` |
| Sklejenie dwóch kolumn | `.str.cat(inna_kolumna, sep=...)` |
| Sklejenie listy (po split) w string | `.str.join(sep)` |
| Element listy po indeksie (bez lambda) | `.str.get(i)` (ujemny indeks też działa) |
| Scalenie tekstów w obrębie grupy | `.groupby("col").agg(", ".join)` |
| Logika łącząca kilka kolumn naraz | `.apply(lambda row: ..., axis=1)` — tu `lambda` jest uzasadniona |
| Wyrównanie do stałej szerokości | `.str.ljust()` / `rjust()` / `center()` |
| Zera wiodące (np. ID o stałej długości) | `.str.zfill(n)` |
| Fragment po POZYCJI (nie separatorze) | `.str.slice(a, b)` / `.str.slice_replace(a, b, nowy)` |
| Powtórzenie całego stringa n razy | `.str.repeat(n)` |
| Dopasowanie od początku / całego tekstu | `.str.match(wzorzec)` / `.str.fullmatch(wzorzec)` |
| Podmiana zależna od dopasowania (nie stały tekst) | `.str.replace(wzorzec, funkcja, regex=True)` |
| Podział po kilku możliwych separatorach naraz | `.str.split(r"[,; ]+")` |
| Opcjonalny fragment we wzorcu | `(?:...)?` w regexie |
| Mapowanie pojedynczych znaków (np. usuwanie diakrytyków) | `.str.translate(str.maketrans(...))` |

**Wniosek:** ten sam motyw co w innych notatkach o pandas — żadna z tych pułapek nie rzuca błędu w oczywisty sposób (poza Pułapką 3). `split(expand=True)` z `NaN`, `partition` z pustymi stringami i `replace()` traktujące `.` jako regex dają wynik, który wygląda na policzony poprawnie, dopóki nie porówna się go z oczekiwaniem.